In [0]:
DECLARE OR REPLACE VARIABLE catalog_use = 'main';
DECLARE OR REPLACE VARIABLE schema_use = 'synthea';

In [0]:
SET VARIABLE catalog_use = :catalog_use;
SET VARIABLE schema_use = :schema_use; 

In [0]:
USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog(), current_schema();

In [0]:
FROM identifier(catalog_use || '.information_schema.routines') |>
WHERE specific_schema = schema_use |>
SELECT specific_catalog, specific_schema, specific_name, routine_type, routine_definition, routine_body, data_type;

In [0]:
SELECT
  databricks_rest_get(
    endpoint => '/2.0/sql'
    ,resource => 'warehouses'
    ,path_parameters => NULL
    ,query_parameters => array('run_as_user_id=')
    ,body => NULL
  )

In [0]:
CREATE OR REPLACE FUNCTION databricks_rest_sql_warehouses_list_warehouses(
  run_as_user_id INTEGER COMMENT 'Service Principal which will be used to fetch the list of warehouses. If not specified, the user from the session header is used.' DEFAULT NULL
)
RETURNS VARIANT 
COMMENT 'Lists all SQL warehouses that a user has manager permissions on.'
LANGUAGE SQL 
RETURN 
SELECT 
  databricks_rest_get(
    endpoint => '/2.0/sql'
    ,resource => 'warehouses'
    ,path_parameters => NULL
    ,query_parameters => array('run_as_user_id=' || run_as_user_id)
    ,body => NULL
  )
;

In [0]:
WITH response as (
  FROM (
    SELECT 
      databricks_rest_sql_warehouses_list_warehouses() AS listing
  )
  ,LATERAL variant_explode(listing:warehouses) as warehouses |>
  SELECT 
    warehouses.pos as warehouses_pos
    ,warehouses.value as warehouses_value
    ,warehouses.value:id::string as warehouse_id
)
FROM response
,LATERAL variant_explode(warehouses_value) |> 
SELECT warehouse_id, key, value |>
PIVOT (first(value) for key in ("auto_resume","auto_stop_mins","channel","cluster_size","creator_id","creator_name","enable_photon","enable_serverless_compute","health","id","jdbc_url","max_num_clusters","min_num_clusters","name","num_clusters","odbc_params","size","spot_instance_policy","state"))

In [0]:
/api/2.0/permissions/warehouses/{warehouse_id}

In [0]:
SELECT
  databricks_rest_get(
    endpoint => '/2.0/permissions'
    ,resource => 'warehouses'
    ,path_parameters => array('04e7540eef92d1fa')
    ,query_parameters => NULL
    ,body => NULL
  )

In [0]:
CREATE FUNCTION databricks_rest_sql_warehouses_get_permissions (
  warehouse_id STRING COMMENT "The SQL warehouse for which to get or manage permissions."
)
RETURNS VARIANT 
COMMENT 'Gets the permissions of a SQL warehouse. SQL warehouses can inherit permissions from their root object.'
LANGUAGE SQL 
RETURN 
SELECT 
  databricks_rest_get(
    endpoint => '/2.0/permissions'
    ,resource => 'warehouses'
    ,path_parameters => array(warehouse_id)
    ,query_parameters => NULL
    ,body => NULL
  )
;

In [0]:
select databricks_rest_sql_warehouses_get_permissions('04e7540eef92d1fa')

In [0]:
SELECT
  databricks_rest_put(
    endpoint => '/2.0/permissions',
    resource => 'warehouses',
    path_parameters => array('04e7540eef92d1fa'),
    query_parameters => NULL,
    body => parse_json('{
      "access_control_list": [
        {
          "permission_level": "CAN_MONITOR",
          "user_name": "matthew.giglia@databricks.com"
        }
      ]
    }')
  )

In [0]:
CREATE OR REPLACE FUNCTION databricks_rest_sql_warehouses_set_permissions (
  warehouse_id STRING COMMENT "The SQL warehouse for which to get or manage permissions."
  ,principal_type STRING COMMENT "One of service_principal, group_name, or user_name."
  ,principal_id STRING COMMENT "When service_principal, the id is the application id of the service principal, for group_name or user_name supply the string name for that group or user respectively."
  ,permission_level STRING COMMENT "Permission level, one of CAN_MANAGE, IS_OWNER, CAN_USE, CAN_MONITOR or CAN_VIEW."
)
RETURNS VARIANT 
COMMENT 'Sets permissions on an object, replacing existing permissions if they exist. Deletes all direct permissions if none are specified. Objects can inherit permissions from their root object.'
LANGUAGE SQL 
RETURN 
SELECT 
  databricks_rest_put(
    endpoint => '/2.0/permissions',
    resource => 'warehouses',
    path_parameters => array('04e7540eef92d1fa'),
    query_parameters => NULL,
    body => parse_json('{"access_control_list": [{"' || principal_type || '": "' || principal_id || '","permission_level": "' || permission_level || '"}]}')
  )
;

In [0]:
SELECT databricks_rest_sql_warehouses_set_permissions(
  warehouse_id => '04e7540eef92d1fa'
  ,principal_type => "user_name"
  ,principal_id => "matthew.giglia@databricks.com"
  ,permission_level => "CAN_USE"
)

In [0]:
SELECT databricks_rest_sql_warehouses_set_permissions(
  warehouse_id => '04e7540eef92d1fa'
  ,principal_type => "user_name"
  ,principal_id => "matthew.giglia@databricks.com"
  ,permission_level => "CAN_MONITOR"
)